In [1]:
!pip install -qU langchain-groq langchain langchain_core "langchain-chroma>=0.1.2" langchain_community sentence_transformers fastembed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73

In [2]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [4]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [5]:
#PROMPT TEMPLATE

# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

In [6]:
from langchain_core.output_parsers import StrOutputParser

In [7]:
import re
from langchain_core.runnables import RunnableLambda

In [8]:
from langchain_core.runnables import RunnableBranch

In [9]:
import os

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma

# from langchain_openai import OpenAIEmbeddings
from sentence_transformers import SentenceTransformer
from langchain.embeddings.base import Embeddings
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

/tmp/ipykernel_1508/1246436890.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# **MetaData Store**

In [10]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Assignment

Build an Automated IT Helpdesk Triage System. The system simulates an IT department by taking an incoming support message and automatically processing it from start to finish without human intervention. It uses an AI "Classifier" node to read the user support message and determine if it is a password issue, a hardware fault, or spam, and then uses a conditional router to send that ticket to the correct specialized AI department. Each department node then applies its own logic—either drafting a tailored response or blocking the message as spam.

IMPORTANT:
The node that handles hardware fault should consult hardware manual (a PDF document)  to identify the solution and give the response

get the pipeline for hardware reaady :)

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

!pip install pypdf -qU

# Load hardware guide
loader = PyPDFLoader("/content/hardware.pdf")
documents = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

# Create embeddings
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

# Create vector store
db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(search_kwargs={"k": 3})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 3.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [13]:
hardware_rag_template = ChatPromptTemplate.from_template("""
You are a hardware support specialist.

Use ONLY the information from the hardware guide below.

If the guide does not contain the answer, say so.

Hardware Guide:
{context}

Support Request:
{support}

Generate a clear, step-by-step guide for the user.
""")

In [14]:
hardware_chain = (
    {
        "context": retriever,
        "support": RunnablePassthrough()
    }
    | hardware_rag_template
    | model
    | StrOutputParser()
)

the rest of it :)

In [15]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a support request classifier that classifies support message: {support} between three types. These types are password issue, hardware issue, or spam."),
        ("human", "Classify the request and give a one word answer. Either password, hardware, or spam."),
    ]
)


chain = prompt_template | model | StrOutputParser()



result = chain.invoke({"support": "my email is filled with junk"})


print(result)

<think>
Okay, let's see. The user wants me to classify their support request into one of three categories: password, hardware, or spam. The message they provided is "my email is filled with junk". Hmm, "junk" in the context of email usually refers to unwanted or unsolicited messages, which is commonly known as spam. Password issues would typically involve problems with login credentials, forgotten passwords, etc. Hardware issues are related to physical devices like computers, printers, etc. Since the user is talking about their email being filled with junk, that's a classic sign of spam. I don't see any mention of login problems or hardware malfunctions here. So the correct classification should be spam.
</think>

spam


In [16]:
classification_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        (
            "human",
            """Classify the type of support required as spam, password, or hardware: {support}
(respond with only one word: spam, password, or hardware)."""
        ),
    ]
)

In [20]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Define missing templates
spam_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant for spam issues."),
        ("human", "Your support request about spam: {support}. We have received your report and will take action."),
    ]
)

password_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant for password issues."),
        ("human", "Your support request about password: {support}. Please visit our password reset page or contact IT for assistance."),
    ]
)

escalate_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant for general support issues."),
        ("human", "Your support request: {support}. This issue requires further escalation. We will get back to you shortly."),
    ]
)


classification_chain = classification_template | model | StrOutputParser()

review = "My printer broke down. its hardware wont work"

# classification
classification = classification_chain.invoke({"support": review})

print("x" * 60)
print("CLASSIFIER OUTPUT:")
print(classification)
print("x" * 60)

# Extract only the last word from the classification output, assuming it's the actual classification
# This handles cases where the model might include its thinking process before the final word.
classification_word = classification.lower().strip().split()[-1]

if "spam" in classification_word:
    print(r"\nSPAM chain\n") # Fixed SyntaxWarning
    result = (spam_feedback_template | model | StrOutputParser()).invoke(
        {"support": review}
    )

elif "password" in classification_word:
    print(r"\nPASSWORD chain\n") # Fixed SyntaxWarning
    result = (password_feedback_template | model | StrOutputParser()).invoke(
        {"support": review}
    )

elif "hardware" in classification_word:
    print(r"\nHARDWARE chain\n") # Fixed SyntaxWarning
    result = hardware_chain.invoke(review)

else:
    print(r"\nESCALATION chain\n") # Fixed SyntaxWarning
    result = (escalate_feedback_template | model | StrOutputParser()).invoke(
        {"support": review}
    )

print("=" * 60)
print("FINAL AGENT RESPONSE:")
print(result)
print("=" * 60)

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
CLASSIFIER OUTPUT:
<think>
Okay, let's see. The user wrote, "My printer broke down. its hardware wont work." I need to classify the type of support required here as spam, password, or hardware.

First, "spam" is about unwanted emails or messages, which doesn't seem relevant here. The user is talking about a printer issue, not anything related to spam. 

Next, "password" support would be if they had trouble with login credentials or account access. The user isn't mentioning anything about a password, so that's probably not it.

Then there's "hardware." The user says their printer broke down and the hardware won't work. That directly refers to a problem with the physical components of the printer. So the issue is with the hardware itself, not software or network issues. 

I should make sure there's no trick here. Sometimes people might mention hardware issues but actually need software help, but the message is straightforward. 